In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/nickwan/creating-player-stats-using-tracking-data/src/small_bench/checkpoints/pre_cell_0.pickle

In [ ]:
%%PandasProfile
### cell 0 ###

factor = 1

# file paths
week1_path = '/home/dias-benchmarks/notebooks/nickwan/creating-player-stats-using-tracking-data/input/nfl-big-data-bowl-2023/week1.csv'
scout_path = '/home/dias-benchmarks/notebooks/nickwan/creating-player-stats-using-tracking-data/input/nfl-big-data-bowl-2023/pffScoutingData.csv'
plays_path = '/home/dias-benchmarks/notebooks/nickwan/creating-player-stats-using-tracking-data/input/nfl-big-data-bowl-2023/plays.csv'
players_path = '/home/dias-benchmarks/notebooks/nickwan/creating-player-stats-using-tracking-data/input/nfl-big-data-bowl-2023/players.csv'

# 1) Read and sample the large tracking file up front to reduce the size of the subsequent join
# 2) Use memory_map and disable low_memory to speed up I/O/type inference
data = pd.read_csv(week1_path, memory_map=True, low_memory=False)
data = data.sample(frac=factor, random_state=0)

# 3) Read scouting data and merge on the explicit key columns for a faster join path
scout = pd.read_csv(scout_path, memory_map=True, low_memory=False)
# assuming the join keys are ['gameId','playId','nflId'] — specifying `on=` avoids pandas scanning for common columns
data = data.merge(scout, how='left', on=['gameId', 'playId', 'nflId'])

# 4) Load the other reference tables (unchanged)
plays = pd.read_csv(plays_path, memory_map=True, low_memory=False)
players = pd.read_csv(players_path, memory_map=True, low_memory=False)

# final shape
data.shape